# Sepsis detection - Multivariate Time-Series Classification problem

In [ ]:
#! python3 -m pip install pandas seaborn numpy matplotlib scikit-learn xgboost scipy "spacy==3.7.5" imbalanced-learn tensorflow

## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense

from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

## Load the dataset and inspects few infromations of the dataset

In [ ]:
dataset = pd.read_csv("./Dataset.csv") # load dataset

In [ ]:
dataset.head(5) # view some rows

In [ ]:
dataset.columns # dataset columns

In [6]:
patientLabels = dataset.groupby("Patient_ID")["SepsisLabel"].max()
trainData, testData = train_test_split(
    patientLabels.index.to_numpy(), test_size=0.30,
    random_state=67, stratify=patientLabels.to_numpy(),
)
trainData = dataset.loc[dataset["Patient_ID"].isin(trainData)].copy()
testData = dataset.loc[dataset["Patient_ID"].isin(testData)].copy()


## Preprocessing the data : split first, input and scale, add padding and drop

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

### Split first (7/1/2) then defining a preprocessor to preprocess data


For numerical features, it will be impute using constant value to avoid leakage and then scale using standard scaler

For categorical features, it will also impute using constant and label encoded it

In [ ]:
# Split train validation test FIRST before touching the data
patient_ids_before_split = []
patient_labels_before_split = []
groups_before_split = dataset.groupby("Patient_ID")

for patient_id, patient_df in groups_before_split:
    label = 0
    for i in patient_df["SepsisLabel"]:
        if i == 1:
            label = 1

    patient_ids_before_split.append(patient_id)
    patient_labels_before_split.append(label)

patient_labels = pd.DataFrame({
    "Patient_ID": patient_ids_before_split,
    "SepsisLabel": patient_labels_before_split
})

In [ ]:
train_patients, other_patients = train_test_split(
    patient_labels, test_size= 0.3, random_state= 67, stratify=patient_labels["SepsisLabel"]
) # 70 train and leave 30 for validation + testing

validation_patients, test_patients = train_test_split(
    other_patients, test_size= 2/3, random_state= 67, stratify=other_patients["SepsisLabel"]
) # from the 30: 10 validation and 20 testing


dataset_train = dataset[dataset["Patient_ID"].isin(train_patients["Patient_ID"])].copy()
dataset_validation = dataset[dataset["Patient_ID"].isin(validation_patients["Patient_ID"])].copy()
dataset_test = dataset[dataset["Patient_ID"].isin(test_patients["Patient_ID"])].copy()

print(len(train_patients), len(validation_patients), len(test_patients))
print(train_patients["SepsisLabel"].value_counts(normalize=True).sort_index())
print(validation_patients["SepsisLabel"].value_counts(normalize=True).sort_index())
print(test_patients["SepsisLabel"].value_counts(normalize=True).sort_index())



In [ ]:
# transforming pipeline for numerical features
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value= 0)), ("scaler", StandardScaler())]
)

# transforming pipeline for catergorical features
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value = "UNKNOWN")), # no data leak hehe
        ("encoder", LabelEncoder()),
    ]
)

# Extract out catergorical and numerical columns
cat_cols = []
num_cols = []
for col_name, dtype in dataset_train.dtypes.items():

    if col_name == "SepsisLabel" or col_name == "Patient_ID":
        continue  # SKIP the label and id of the patient

    if np.issubdtype(dtype, np.number):
        num_cols.append(col_name)
    else:
        cat_cols.append(col_name)


# define precessors
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols,
        ),
        
    ],
    remainder="passthrough" # for the label and id
)

columns = dataset.columns

data_train_processed = preprocessor.fit_transform(dataset_train) # ONLY fit using train because no leak
data_validation_processed = preprocessor.transform(dataset_validation)
data_test_processed = preprocessor.transform(dataset_test)

dataset_train_reconstruct = pd.DataFrame(data_train_processed, columns= columns) # reconstruct back to time-series preocessing
dataset_validation_reconstruct = pd.DataFrame(data_validation_processed, columns= columns)
dataset_test_reconstruct = pd.DataFrame(data_test_processed, columns= columns)

In [ ]:
print(dataset_train_reconstruct.shape) # correct dimensions
print(dataset_validation_reconstruct.shape)
print(dataset_test_reconstruct.shape)

### Time-series processing: 

1. Find 95% of TRAIN patients window lenght (How many hours 95% percentile patients data last)

2. Drop all patients above that. For patients who above 95%, a sliding window technique can be apply where it is expensive but less expensive to train (trade-off).

3. Transform the problem to detection problem. As a patient can have Sepsis in multiple hours but not others. If there is sepsis in any hour, the patient is classified as sepsis

NOTE: justification to remain time-series problem rather time-step prediction ( simple supervised problem ): temporal information can carries informations about the patient information hence having full window retains useful information

In [ ]:
groups_train = dataset_train_reconstruct.groupby("Patient_ID") # Group patients records
groups_validation = dataset_validation_reconstruct.groupby("Patient_ID")
groups_test = dataset_test_reconstruct.groupby("Patient_ID")

hours =[] # how many hours is each patient records last

# Analysis of the patient time lentgh
for patient_id, patient_df in groups_train:
    hours.append(patient_df.shape[0])

window = np.percentile(hours,95) # 95% percentile ONLY from train data
window # 58 hours


In [ ]:
# preprocessing : drop all > 95% percentile hours and add padding
def add_padding(groups,window):

    target_var = []
    transformed_sequence = []


    for patient_id, patient_df in groups:

        #extract the target variable
        # if the patient have sepsis, they will have at least 1 row to be 1
        label = 0
        for i in patient_df["SepsisLabel"]:
            if i == 1:
                label = 1

        patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])

        if patient_df.shape[0] > window:
            #im lazy sorry:(
            continue

        target_var.append(label)
        #add padding
        patient_df = patient_df.sort_values("Hour")
        
        num_feature = patient_df.shape[1] # should be 43 anyway


        #change to np
        patient_np = patient_df.to_numpy()
        num_padding_rows = int(window) - patient_df.shape[0]
        all_zeros = np.zeros(num_feature)
        new_patient = []
        for i in range (num_padding_rows):
            new_patient.append(all_zeros)
        for hour in patient_np:
            new_patient.append(hour)

        transformed_sequence.append(new_patient)

    transformed_sequence = np.asarray(transformed_sequence)
    target_var = np.asarray(target_var)
    return transformed_sequence,target_var

# Add padding to all set: train , validation, test
X_train,y_train = add_padding(groups_train,window)
X_validation,y_validation = add_padding(groups_validation,window)
X_test,y_test = add_padding(groups_test,window)

print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)

In [ ]:
X_train[0:5] # Expect a 3D array 

In [ ]:
y_train # this is our ground truth for train

In [ ]:
target_var_train_np = np.asarray(y_train) # transform to np array
target_var_validation_np = np.asarray(y_validation)
target_var_test_np = np.asarray(y_test)

In [ ]:
print(target_var_train_np.shape) # check shape
print(target_var_validation_np.shape)
print(target_var_test_np.shape)

In [ ]:
# Already split 7/1/2 before preprocessing but check for sure
print(X_train.shape)
print(X_validation.shape)
print(X_test.shape)
print(len(y_train))
print(len(y_validation))
print(len(y_test))

In [17]:
print(len([one for one in y_train if one == 0])) # How non-sepsis sepsis is distributed (Data is VERY IMBALANCE)

### Upsampling by SMOTE
1. Flatten out to SMOTE

2. Apply SMOTE (SMOTE perturbs and use KKN to get the neighbours). This upsampling under-represented label

3. Fit SMOTE and yeah! ONLY TRAIN THOUGH validation and testing remains real

In [ ]:
#SMOTE sampling X_train

#Flatten to 2D

num_sample, num_time, num_feature = X_train.shape

X_train_flatten = X_train.reshape(num_sample, num_time*num_feature) #Flatten to do SMOTE
#SMOTE time
s = SMOTE(k_neighbors=10, random_state= 67)
X_train_sampled_flatten , y_train_sampled = s.fit_resample(X_train_flatten,y_train)
X_train_sampled = X_train_sampled_flatten.reshape(-1, num_time, num_feature) # reshape back to 3D

X_train_sampled.shape

In [19]:
# check shapes
print(len(y_train_sampled))

print(len([one for one in y_train_sampled if one == 1]))
print(len([one for one in y_train_sampled if one == 0]))

# data is balanced now

In [20]:
# recontruct back to np array
X_train_sampled = np.asarray(X_train_sampled, dtype=np.float32)
y_train_sampled = np.asarray(y_train_sampled, dtype=np.float32)
X_train = np.asarray(X_train, dtype=np.float32)
y_train = np.asarray(y_train, dtype=np.float32)
X_validation = np.asarray(X_validation, dtype=np.float32)
y_validation = np.asarray(y_validation, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

### Baseline: RNN (simple small) - Note: No pass this baseline on Recall Sepsis -> Try again

In [ ]:
# Baseline SimpleRNN for the patient sequences
import random
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

num_time_steps, num_features = X_train_sampled.shape[1], X_train_sampled.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train_sampled,
    y_train_sampled,
    validation_data=(X_validation, y_validation),
    epochs=10,
    batch_size=100,
    verbose=2,
)

test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

In [ ]:

num_time_steps, num_features = X_train.shape[1], X_train.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=10,
    batch_size=100,
    verbose=2,
)



GRU

In [51]:
from tensorflow.keras.layers import GRU

model =tf.keras.Sequential()
model.add(GRU(16, input_shape=X_train_sampled.shape[1:])) 
model.add(Dense(1, activation="sigmoid")) 
model.add(Dense(1, activation="sigmoid")) 

model.compile(loss='binary_crossentropy', optimizer='adam')
model.fit(X_train, y_train, class_weight= {0:0.1,1:5}, epochs=20, batch_size=100, verbose=2)



Epoch 1/20


C:\Users\Samuel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


271/271 - 2s - 9ms/step - loss: 0.2336
Epoch 2/20
271/271 - 2s - 6ms/step - loss: 0.2059
Epoch 3/20
271/271 - 2s - 6ms/step - loss: 0.1896
Epoch 4/20
271/271 - 2s - 6ms/step - loss: 0.1773
Epoch 5/20
271/271 - 2s - 6ms/step - loss: 0.1668
Epoch 6/20
271/271 - 2s - 6ms/step - loss: 0.1581
Epoch 7/20
271/271 - 1s - 6ms/step - loss: 0.1504
Epoch 8/20
271/271 - 2s - 6ms/step - loss: 0.1436
Epoch 9/20
271/271 - 2s - 6ms/step - loss: 0.1380
Epoch 10/20
271/271 - 2s - 6ms/step - loss: 0.1331
Epoch 11/20
271/271 - 2s - 6ms/step - loss: 0.1288
Epoch 12/20
271/271 - 2s - 6ms/step - loss: 0.1249
Epoch 13/20
271/271 - 2s - 6ms/step - loss: 0.1222
Epoch 14/20
271/271 - 2s - 6ms/step - loss: 0.1183
Epoch 15/20
271/271 - 2s - 6ms/step - loss: 0.1154
Epoch 16/20
271/271 - 2s - 6ms/step - loss: 0.1139
Epoch 17/20
271/271 - 2s - 6ms/step - loss: 0.1114
Epoch 18/20
271/271 - 2s - 6ms/step - loss: 0.1090
Epoch 19/20
271/271 - 2s - 6ms/step - loss: 0.1069
Epoch 20/20
271/271 - 2s - 6ms/step - loss: 0.1061


In [52]:
test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

              precision    recall  f1-score   support

   No sepsis     0.9898    0.8435    0.9108     11024
      Sepsis     0.2177    0.8333    0.3452       576

    accuracy                         0.8430     11600
   macro avg     0.6037    0.8384    0.6280     11600
weighted avg     0.9514    0.8430    0.8827     11600

ROC-AUC: 0.9002


In [ ]:
import joblib
model.save("./baseline_rnn.keras")
joblib.dump(preprocessor, "./preprocessor")

In [ ]:
# Load the saved model and preprocessing pipeline
import joblib
import tensorflow as tf

model = tf.keras.models.load_model("./baseline_rnn.keras")
preprocessor = joblib.load("./preprocessor")

In [ ]:
dataset_reconstruct

In [ ]:
dummy_num_rows = 10
# The saved preprocessor was fitted with all original columns, so use zero placeholders
# for the excluded columns only while transforming, then keep them for add_padding.
dummy_raw = pd.DataFrame(
    np.zeros((dummy_num_rows, len(dataset.columns))),
    columns=dataset.columns,
)
dummy_data = pd.DataFrame(
    preprocessor.transform(dummy_raw),
    columns=dataset.columns,
)

print(f"Dummy patient shape after preprocessing: {dummy_data.shape}")

def add_padding(patient_df, window):
    # Assumes that the patient has fewer rows than the window.
    for i in patient_df["SepsisLabel"]:
        print(i)
    patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])
    patient_df = patient_df.sort_values("Hour")

    num_feature = patient_df.shape[1]
    patient_np = patient_df.to_numpy()
    num_padding_rows = int(window) - patient_df.shape[0]
    all_zeros = np.zeros(num_feature)
    new_patient = []
    count = 0
    for i in range(num_padding_rows):
        new_patient.append(all_zeros)
        count += 1
    for hour in patient_np:
        new_patient.append(hour)
        count += 1
    new_patient_np = np.asarray(new_patient)
    print(count)
    print(len(new_patient))
    return new_patient_np

X_infer = add_padding(dummy_data, 58)[None, ...]
print(X_infer.shape)
y = model.predict(X_infer)
if y[0][0] > 0.5:
    print("Sepsis")

if y[0][0] < 0.5:
    print("No Sepsis")
print(y)